# Ghaderi, Crochet & Petersen (2026) from dandiset 001530: cluster removal, post-whisker decoding and clustering

**Paper.** Ghaderi P, Crochet S, Petersen CCH (2026), *Nature Communications* 17, 5982,
[doi:10.1038/s41467-026-73622-y](https://doi.org/10.1038/s41467-026-73622-y); claims written from the
version of record (PMC13346651), main text, legends and Methods.

**Dandiset.** [DANDI:001530](https://dandiarchive.org/dandiset/001530/0.260605.0601), version
0.260605.0601, the only published version (the draft has the same assets and was last modified
with the publication); the paper cites no version. The data check, the trial and unit selection
and the claims on behaviour, optogenetics and single neurons are in `reproduce_ghaderi_2026.ipynb`;
this notebook loads the same sessions through the same helper.

**Outside inputs.** `data_helpers/Area_list.mat` (CCF labels of each area) and
`processed_data/Data_Clustering.mat` (the paper's cluster labels, linked by claim
`cluster-label-link` in the first notebook), from Zenodo doi:10.5281/zenodo.17973874.

**Run size.** The paper's decoding repeats each fit over 100 neuron subsets (Fig. 5b) or 5 runs
(Fig. 7d), and its GMM uses 1,000 initialisations per k. The Methods' grid search fits 36 SVMs for every
one of the code's decoder (7 values of C in an inner 5-fold search, then the refit), and the run
times printed below put a full run of either reading beyond what fits here, so both are run with
fewer repetitions (constants in the next cell); each section prints what was run, its run time and
the estimate for the full run.

| Claim | Here |
|---|---|
| fig5b-context-decoding-delay, fig5b-decoding-time-course | `decoding_delay.ipynb` |
| fig5c-cluster-removal | tested (cluster labels from Zenodo) |
| fig7d-context-decoding-after-whisker | tested, fewer runs |
| fig4-29-clusters | tested, fewer GMM initialisations |
| fig5d-null-space-decoding | blocked (see the record) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import ghaderi as g

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

N_JOBS = 4
REPS_CODE = 5           # neuron subsets per session, code's decoder (paper: 100)
REPS_TC = 3             # the same for 5 to 45 neurons (time course)
REPS_METHODS = 1        # neuron subsets per session, Methods' grid search (paper: 100)
BASE_STEP_METHODS = 5   # Methods' reading: every 5th baseline bin of -1 to 0 s
RUNS_7D = 5             # Fig. 7d runs of the code's decoder (code: 5)
GMM_INITS = 5           # GMM initialisations per k (paper: 1,000)

started = time.time()
S = g.load_sessions(workers=4)
print(f"dandiset {g.DANDISET_ID} version {g.DANDISET_VERSION}: loaded {len(S)} sessions in {time.time() - started:.0f} s")

In [ ]:
rows = []
for a in g.AREAS:
    for s, cols in g.area_sessions(S, a, 5):
        q = g.quiet_mask(s)
        rows.append({"area": a, "session": s["session"], "subject": s["subject"], "neurons": len(cols),
                     "quiet Hit": int((g.trial_mask(s, 1, True) & q).sum()),
                     "quiet Nogo CR": int((g.trial_mask(s, 3, False) & q).sum())})
dsess = pd.DataFrame(rows)
print(dsess.to_string(index=False))
print("\nArea-sessions with at least 5 neurons:", dsess.groupby("area").size().loc[g.AREAS].to_dict())
print("with at least 50 neurons:", dsess[dsess.neurons >= 50].groupby("area").size().loc[g.AREAS].to_dict(),
      "(paper, Fig. 5b: A1 8, wS1 17, wS2 19, wM2 18, ALM 16)")
print("mice with at least 50 neurons:", dsess[dsess.neurons >= 50].groupby("area").subject.nunique().loc[g.AREAS].to_dict(),
      "(paper: 6, 9, 9, 8, 6)")
print("Spike counts per 10 ms bin, range:", min(int(s["c10"].min()) for s in S if s["c10"].size), "to", max(int(s["c10"].max()) for s in S if s["c10"].size))

The sessions with at least 50 neurons per area match the paper's n for Fig. 5b (8, 17, 19, 18, 16
sessions); quiet Hit and Nogo CR trials per session are listed above, some as few as 7.

In [ ]:
DELAY = g.bin_index(0.8, 1.0)
BASE = g.bin_index(-1.0, 0.0)
BASE_CODE = g.bin_index(-0.8, 0.0)
CUE_A, CUE_B = g.bin_index(0.01, 0.03), g.bin_index(0.01, 0.2)


def session_classes(s, cols, rule="median", counts="c10"):
    q = g.quiet_mask(s, rule)
    X = s[counts][:, :, cols]
    return X[g.trial_mask(s, 1, True) & q], X[g.trial_mask(s, 3, False) & q]

## fig5c-cluster-removal

> Removing cluster 11 reduces late-delay context decoding in wM2 (by 3.8 +/- 1.0%) and ALM (4.5 +/- 1.4%)
> with no significant effect in A1, wS1 and wS2; removing cluster 25 also reduces it in wM2 (1.6 +/-
> 0.6%) and ALM (2.0 +/- 0.6%); removing any other cluster does not significantly decrease decoding.

All neurons of each area-session (at least 5), decoded once with all and once without the neurons of
a cluster, with the same random state so that the two differ only by the neurons removed; the
decrease is with minus without (the criterion's change, without minus with, with its sign turned so
that a positive value is a loss). Code's decoder for all 29 clusters; the Methods' grid search for
clusters 11 and 25. Cluster labels from Data_Clustering.mat, linked as in the first notebook.

In [ ]:
CL = g.cluster_labels()
GID = g.global_ids(S)
unit_cluster = {GID[gi]: cl for gi, cl in zip(CL["global_id"], CL["cluster"]) if gi in GID}
print("labelled units linked:", len(unit_cluster), "of", len(CL["cluster"]))
ctasks, ctasks_mean = [], []
sess_index = {s["session"]: i for i, s in enumerate(S)}
for a in g.AREAS:
    for s, cols in g.area_sessions(S, a, 5):
        i = sess_index[s["session"]]
        labs = np.array([unit_cluster.get((i, int(u)), 0) for u in s["keep"][cols]])
        X1, X2 = session_classes(s, cols)
        ctasks.append((a, s["session"], X1, X2, labs))
        X1, X2 = session_classes(s, cols, rule="mean")
        ctasks_mean.append((a, s["session"], X1, X2, labs))


def removal(X1, X2, labs, clusters, method, seed):
    full = g.decode_session(X1, X2, DELAY, None, 1, method, seed).mean()
    out = {}
    for c in clusters:
        drop = labs == c
        if drop.any() and (~drop).sum() >= 1:
            out[c] = full - g.decode_session(X1, X2, DELAY, None, 1, method, seed, drop=drop).mean()
        else:
            out[c] = 0.0
    return full, out


# Runs: the main one (seed set 0), and for the code's decoder two more seed sets (the verifier asked
# for the spread over random draws) and the code's quiet rule. The grid search is run once (time).
runs = {}
t0 = time.time()
for r in range(3):
    runs[("b, code", f"set {r}")] = Parallel(n_jobs=N_JOBS)(delayed(removal)(t[2], t[3], t[4], range(1, 30), "code", 10000 * r + 5000 + i)
                                                           for i, t in enumerate(ctasks))
runs[("b, code", "code MAD")] = Parallel(n_jobs=N_JOBS)(delayed(removal)(t[2], t[3], t[4], range(1, 30), "code", 5000 + i)
                                                       for i, t in enumerate(ctasks_mean))
t_code = time.time() - t0
print(f"code's decoder, 29 clusters, 4 runs: {t_code:.0f} s")
print("The paper's decoder for this figure (Decoding_clusters_random_dropout.m) is the same as fig5b's, without neuron subsets.")
t0 = time.time()
for r in range(1):
    runs[("a, Methods", f"set {r}")] = Parallel(n_jobs=N_JOBS)(delayed(removal)(t[2], t[3], t[4], (11, 25), "methods", 10000 * r + 6000 + i)
                                                              for i, t in enumerate(ctasks))
t_meth = time.time() - t0
print(f"Methods' grid search, clusters 11 and 25: {t_meth:.0f} s per run; all 29 clusters would take about {t_meth * 30 / 3 / 60:.0f} min per run")
rows = []
for (reading, run), res in runs.items():
    for t, (full, out) in zip(ctasks, res):
        for c, d in out.items():
            rows.append({"reading": reading, "run": run, "area": t[0], "session": t[1], "cluster": c, "full": full, "decrease": d,
                         "removed": int((t[4] == c).sum())})
drop_df = pd.DataFrame(rows)
# The mean over the three seed sets per session, as a run with less Monte-Carlo noise.
avg = drop_df[drop_df.run.str.startswith("set")].groupby(["reading", "area", "session", "cluster"], as_index=False)[["full", "decrease", "removed"]].mean()
avg["run"] = "mean of sets"
drop_df = pd.concat([drop_df, avg], ignore_index=True)
summ = []
for (reading, run, c, a), sub in drop_df.groupby(["reading", "run", "cluster", "area"]):
    nz = sub.decrease.values
    p_ = stats.wilcoxon(nz).pvalue if np.any(nz != 0) else 1.0
    summ.append({"reading": reading, "run": run, "cluster": c, "area": a, "sessions": len(sub), "sessions with it": int((sub.removed > 0).sum()),
                 "decrease": nz.mean(), "SEM": nz.std(ddof=1) / np.sqrt(len(nz)), "p": p_})
summ = pd.DataFrame(summ)
summ["p_FDR"] = summ.groupby(["reading", "run", "cluster"]).p.transform(lambda p: g.bh_fdr(p.values))
summ["significant decrease"] = (summ.p_FDR < 0.05) & (summ.decrease > 0)
print("\nMain run (seed set 0):")
print(summ[summ.cluster.isin([11, 25]) & (summ.run == "set 0")].round(4).to_string(index=False))
others = summ[(summ.reading == "b, code") & (summ.run == "set 0") & ~summ.cluster.isin([11, 25]) & summ["significant decrease"]]
print("\nOther cluster-area pairs with a significant decrease (code's decoder, set 0):", others[["cluster", "area", "decrease", "p_FDR"]].round(4).values.tolist())
print("Full-population delay accuracy (code, set 0), mean per area:",
      drop_df[(drop_df.reading == "b, code") & (drop_df.run == "set 0")].groupby("area").full.mean().round(2).loc[g.AREAS].to_dict())

In [ ]:
# Every run: clusters 11 and 25 in wM2 and ALM (decrease, p_FDR), cluster 11 elsewhere, other clusters.
exp5c = {(11, "wM2"): (3.8, 1.0), (11, "ALM"): (4.5, 1.4), (25, "wM2"): (1.6, 0.6), (25, "ALM"): (2.0, 0.6)}
rows = []
for (reading, run), sub in summ.groupby(["reading", "run"], sort=False):
    s_ = sub.set_index(["cluster", "area"])
    row = {"reading": reading, "run": run}
    four = True
    for (c, a), (v, tol) in exp5c.items():
        row[f"c{c} {a}"] = f"{s_.loc[(c, a), 'decrease']:.2f} (p_FDR {s_.loc[(c, a), 'p_FDR']:.3f})"
        four &= bool(abs(s_.loc[(c, a), "decrease"] - v) <= tol and s_.loc[(c, a), "significant decrease"])
    row["four within and significant"] = four
    row["c11 significant in A1, wS1, wS2"] = [a for a in ("A1", "wS1", "wS2") if s_.loc[(11, a), "significant decrease"]]
    oth = sub[~sub.cluster.isin([11, 25]) & sub["significant decrease"]]
    row["other pairs significant"] = ("not run" if reading.startswith("a") else
                                      [(int(c), a, round(d, 2)) for c, a, d in oth[["cluster", "area", "decrease"]].values])
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nRange of the per-session decreases (code, set 0):",
      round(drop_df[(drop_df.reading == "b, code") & (drop_df.run == "set 0")].decrease.min(), 2), "to",
      round(drop_df[(drop_df.reading == "b, code") & (drop_df.run == "set 0")].decrease.max(), 2), "points")

**Result: blocked (methods underspecified), the verified outcome; a change to not reproduced is
proposed.** Code's nu-SVC, seed set 0: removing cluster 11 lowers delay decoding in wM2 by 4.61 and in
ALM by 4.45 points (p_FDR 0.0009 and 0.0002) and not significantly in A1, wS1 or wS2; cluster 25 by
1.30 and 1.70 (p_FDR 0.048 in both), all four within tolerance; one other pair decreases significantly
(cluster 24 in A1, 0.49, p_FDR 0.029): partially reproduced. That is the only run of the code's decoder
that gives partial. In seed sets 1 and 2 cluster 25 is not significant (p_FDR 0.075 to 0.234); the mean
of the three sets has the four values within tolerance and significant but three other pairs
significant (cluster 2 in wM2, cluster 13 in wM2 and ALM); and with the code's quiet rule cluster 11 is
also significant in wS2, which the claim excludes (and cluster 26 in wM2): not reproduced in 4 of the 5
runs. Set 0 has no claim to be the main run; it was the first. Methods' grid search (one run, clusters
11 and 25 only): cluster 11 gives 4.17 and 4.72 points (significant) and a small significant decrease
in wS1 (0.20, p_FDR 0.030), and cluster 25 gives 1.31 and 1.91 points, not significant after FDR
(p_FDR 0.085): not reproduced. So the Methods' reading and the code's reading in its less noisy runs
(the mean of the sets) give not reproduced; the record proposes that outcome for a third round. The
grid search for all 29 clusters (about 74 min per run by the time printed above) and its spread over
seeds were not run.

## fig7d-context-decoding-after-whisker

> Decoding context from baseline-subtracted activity in 5 ms bins after the whisker stimulus,
> significant decoding appears transiently in A1 and wS2 at 15 ms for one bin, and persistently in wM2
> from 20 ms and in ALM from 25 ms after the stimulus.

As the criterion (code, Decoding_afterwhisker_runX.m), all neurons, 5 ms bins labelled 0.955 to
1.100 s; each trial minus its own count in one baseline bin, repeated for each of the 10 bins from
0.955 to 1.000 s, accuracies averaged; code's decoder, RUNS_7D run(s) (set above). The Methods' grid search is
run with two of the baseline bins (0.955 and 1.000 s). Per bin, Wilcoxon against the mean accuracy
over 0.955 to 1.0 s, FDR over the 20 bins after the stimulus within each area.

In [ ]:
lab5 = g.T5_START + g.W5 * np.arange(1, g.N5 + 1)


def decode_after_whisker(X1, X2, base_bins, method, seed, runs):
    rng_seeds = np.random.default_rng(seed).integers(2 ** 31, size=(runs, len(base_bins)))
    acc = np.zeros(g.N5)
    for r in range(runs):
        for k, b in enumerate(base_bins):
            Y1 = X1.astype(float) - X1[:, b:b + 1].astype(float)
            Y2 = X2.astype(float) - X2[:, b:b + 1].astype(float)
            acc += g.decode_session(Y1, Y2, range(g.N5), None, 1, method, int(rng_seeds[r, k]))
    return acc / (runs * len(base_bins))


wtasks = []
for a in g.AREAS:
    for s, cols in g.area_sessions(S, a, 5):
        X1, X2 = session_classes(s, cols, counts="c5")
        wtasks.append((a, s["session"], X1, X2))
acc7 = {}
t0 = time.time()
acc7["b, code, 5 runs"] = Parallel(n_jobs=N_JOBS)(delayed(decode_after_whisker)(t[2], t[3], range(10), "code", 7000 + i, RUNS_7D)
                                                  for i, t in enumerate(wtasks))
print(f"code's decoder, {RUNS_7D} runs: {time.time() - t0:.0f} s")
t0 = time.time()
acc7["a, Methods, 1 run, baseline bins 0.955 and 1.000 s"] = Parallel(n_jobs=N_JOBS)(
    delayed(decode_after_whisker)(t[2], t[3], (0, 9), "methods", 8000 + i, 1) for i, t in enumerate(wtasks))
t_m = time.time() - t0
print(f"Methods' grid search, 2 baseline bins, 1 run: {t_m:.0f} s; all 10 bins would take about {t_m * 5 / 60:.0f} min per run "
      "by this run's time (a run with all 10 bins did not finish in 53 min while preparing this revision)")

POST = np.arange(10, 30)


def onsets_of(accs, verbose):
    onsets, singles = {}, {}
    for a in g.AREAS:
        A = np.array([acc for t, acc in zip(wtasks, accs) if t[0] == a])
        base = A[:, :10].mean(axis=1)
        p = np.array([stats.wilcoxon(A[:, i], base).pvalue for i in POST])
        q = g.bh_fdr(p)
        sig = (q < 0.05) & (A[:, POST].mean(axis=0) > base.mean())
        onsets[a] = next((round(lab5[POST[i]], 3) for i in range(len(POST)) if sig[i:].all()), None)
        singles[a] = [round(lab5[POST[i]], 3) for i in np.flatnonzero(sig)]
        if verbose:
            print(f"  {a} ({len(A)} sessions): mean accuracy by bin {np.round(A.mean(axis=0)[POST], 1).tolist()}")
            print(f"     significant bins {singles[a]}; persistent run from {onsets[a]}")
    ok = (onsets["wM2"] is not None and 1.0149 <= onsets["wM2"] <= 1.0251 and onsets["ALM"] is not None
          and 0.0049 <= onsets["ALM"] - onsets["wM2"] <= 0.0101
          and all(onsets[a] is None or onsets[a] >= 1.0299 for a in ("A1", "wS1", "wS2")))
    precedes = onsets["wM2"] is not None and onsets["ALM"] is not None and onsets["wM2"] < onsets["ALM"]
    return onsets, singles, ok, precedes


rows = []
for key, accs in acc7.items():
    print(f"\nReading {key}")
    onsets, singles, ok, precedes = onsets_of(accs, verbose=True)
    rows.append({"variant": key, **{f"onset {a}": onsets[a] for a in g.AREAS}, "criterion holds": ok, "wM2 before ALM": precedes,
                 "significant bins at 1.015 (A1, wS2)": [a for a in ("A1", "wS2") if 1.015 in singles[a]]})
print()
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (methods underspecified).** With the code's nu-SVC averaged over 5 runs (the
code's number), the persistent run of significant bins starts at 1.020 s in wM2 and 1.025 s in ALM,
5 ms later, and in A1, wS1 and wS2 only at 1.085, 1.060 and 1.060 s: the criterion holds (reproduced);
A1 also has a single significant bin at 1.015 s, as the paper reports. With the Methods' grid search
(one run, baseline bins 0.955 and 1.000 s), wM2's run starts at 1.040 s, one bin after ALM's at 1.035 s,
so wM2 does not precede ALM (not reproduced). The readings give different outcomes, which the
criterion blocks. The grid search with all 10 baseline bins, estimated above at about 58 min per run,
was started while preparing this revision and had not finished after 53 min, so it was not run here;
this Methods result rests on 2 of the 10 baseline bins and one run.

## fig4-29-clusters

> Clustering all 10,294 neurons on their average activity in the five correct trial types (PCA,
> spectral embedding, Gaussian mixture model) gives 29 clusters at the minimum of the Bayesian
> Information Criterion.

As the criterion. Methods: 10 ms PSTHs of the five correct types over -1 to 2 s, baseline subtracted,
concatenated (1,500 bins), divided by their range, mean-centred, 17 PCs (the left singular vectors,
as Data_Clustering.ipynb), Gaussian similarity with sigma giving a mean similarity of 0.5,
eigenvectors 2 to 17 of the normalised Laplacian, diagonal GMM for the code's k values. The code's
variant: 100 ms bins, quiet by whisker speed only, exp(-d / 0.07975). The shuffle rule for the number
of PCs is not run (1,000 PCAs of the full matrix).

In [ ]:
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
from sklearn.mixture import GaussianMixture

# The code's choices this section follows, as printed from Data_Clustering.ipynb at the pinned commit
# (row centring, unscaled left singular vectors, the code's kernel, eigenvectors 2 to 17, the k grid,
# diagonal covariance, reg_covar 0, 1,000 initialisations).
print(g.code_lines("processed_data_code/Data_Clustering.ipynb",
                   r"temp_ND_pdf = temp_ND_pdf - np.mean|np.linalg.svd\(temp_ND_pdf|S_Both = np.exp|sig = 0.07975|"
                   r"spectral_Data = V_Both|k = list\(range\(10|Sigma = \['diag'\]|RegularizationValue = 0|MaxIter = 1000|"
                   r"Replicates =1000|random_state=Seed"))



def cluster_matrix(rebin, signals):
    rows, n_skipped = [], 0
    for a in g.AREAS:
        for s, cols in g.area_sessions(S, a, 5):
            q = g.quiet_mask(s, "median", signals)
            masks = [g.correct_mask(s, t_) & q for t_ in range(1, 6)]
            if any(m.sum() < 3 for m in masks):
                n_skipped += len(cols)
                continue
            parts = []
            for m in masks:
                x = s["c10"][m][:, :, cols].astype(float).mean(axis=0) / g.W10
                if rebin > 1:
                    x = x.reshape(g.N10 // rebin, rebin, -1).mean(axis=1)
                nb = x.shape[0] // 3
                parts.append(x - x[:nb].mean(axis=0))
            rows.append(np.concatenate(parts, axis=0).T)
    X = np.concatenate(rows, axis=0)
    rng_ = X.max(axis=1) - X.min(axis=1)
    X = X[rng_ > 0] / rng_[rng_ > 0, None]
    # Each neuron's row centred, as Data_Clustering.ipynb (A4_spectral_embedding) does.
    return X - X.mean(axis=1, keepdims=True), n_skipped


def embed(X, n_pcs, kernel, sigma=None):
    # The unscaled left singular vectors as the PCs, as A4_spectral_embedding does.
    U = np.linalg.svd(X, full_matrices=False)[0][:, :n_pcs]
    D = squareform(pdist(U)).astype(np.float32)
    if kernel == "gauss":
        lo, hi = 1e-4, 10.0
        for _ in range(40):
            sig = np.sqrt(lo * hi)
            m = np.exp(-D ** 2 / (2 * sig ** 2)).mean()
            lo, hi = (lo, sig) if m > 0.5 else (sig, hi)
        sigma = np.sqrt(lo * hi)
        W = np.exp(-D ** 2 / (2 * sigma ** 2))
    else:
        W = np.exp(-D / sigma)
    mean_sim = float(W.mean())
    d = 1 / np.sqrt(W.sum(axis=1))
    W *= d[:, None]
    W *= d[None, :]
    vals, vecs = eigsh(W.astype(np.float64), k=17, which="LA")
    order = np.argsort(-vals)       # largest of D^-1/2 W D^-1/2 = smallest of the Laplacian
    vecs = vecs[:, order]
    return vecs[:, 1:17], sigma, mean_sim


K_VALUES = list(range(10, 25, 5)) + list(range(25, 45)) + list(range(45, 56, 5))
GMM_SEEDS = {"Methods": (0, 1, 2, 3), "code": (0, 1)}


def bic_scan(E, seed, reg):
    """BIC per k of a diagonal GMM with GMM_INITS initialisations (code, A6_GMM_grid_search:
    reg_covar 0, max_iter 1000). The best of several initialisations is the one of highest
    likelihood, which for a fixed k is the one of lowest BIC, so the minimum over seeds of the
    BIC per k is the scan with GMM_INITS x seeds initialisations."""
    import threadpoolctl
    out = []
    with threadpoolctl.threadpool_limits(1):
        for k in K_VALUES:
            gm = GaussianMixture(k, covariance_type="diag", max_iter=1000, n_init=GMM_INITS, random_state=seed, reg_covar=reg).fit(E)
            out.append({"k": k, "seed": seed, "reg_covar": reg, "BIC": gm.bic(E), "converged": gm.converged_})
    return pd.DataFrame(out)


t0 = time.time()
clu, best_k = {}, []
for variant, rebin, signals, kernel, sigma in (("Methods", 1, ("whisker_speed", "jaw_movement"), "gauss", None),
                                               ("code", 10, ("whisker_speed",), "exp", 0.07975)):
    X, skipped = cluster_matrix(rebin, signals)
    E, sig, ms = embed(X, 17, kernel, sigma)
    jobs = [(sd, 0.0) for sd in GMM_SEEDS[variant]]
    scans = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(bic_scan)(E, sd, reg) for sd, reg in jobs), ignore_index=True)
    clu[variant] = scans
    print(f"{variant}: {X.shape[0]} neurons x {X.shape[1]} bins (neurons skipped: {skipped}); sigma {sig:.4f}, "
          f"mean similarity {ms:.3f}; variance of the embedding columns {E.var(axis=0).min():.2e} to {E.var(axis=0).max():.2e}; "
          f"not converged: {int((~scans.converged).sum())} of {len(scans)} fits; {time.time() - t0:.0f} s")
    for (sd, reg), sc in scans.groupby(["seed", "reg_covar"]):
        k_ = int(sc.loc[sc.BIC.idxmin(), "k"])
        best_k.append({"variant": variant, "GMM": f"seed {sd}, reg_covar {reg:g}, {GMM_INITS} initialisations", "BIC minimum at k": k_})
    pooled = scans[scans.reg_covar == 0].groupby("k").BIC.min()
    best_k.append({"variant": variant, "GMM": f"best of the {GMM_INITS * len(GMM_SEEDS[variant])} initialisations (seeds pooled), reg_covar 0",
                   "BIC minimum at k": int(pooled.idxmin())})
    print("   pooled BIC by k:", {int(k): round(v) for k, v in pooled.items()})
best_k = pd.DataFrame(best_k)
print()
print(best_k.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for variant, scans in clu.items():
    pooled = scans[scans.reg_covar == 0].groupby("k").BIC.min()
    ax.plot(pooled.index, pooled - pooled.min(), "-o", ms=3, label=f"{variant}, best of {GMM_INITS * len(GMM_SEEDS[variant])}")
ax.axvline(29, color="k", lw=0.5)
ax.set_xlabel("k")
ax.set_ylabel("BIC - min")
ax.legend()
plt.show()
for variant, scans in clu.items():
    pooled = scans[scans.reg_covar == 0].groupby("k").BIC.min()
    print(f"{variant}: plotted range 0 to {pooled.max() - pooled.min():.0f}; BIC over k = 30 to 44 spans "
          f"{pooled.loc[30:44].max() - pooled.loc[30:44].min():.0f}; BIC at k = 29 is {pooled.loc[29] - pooled.min():.0f} above the minimum")
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

**Result: not reproduced.** Following the Methods (10 ms PSTHs, Gaussian similarity with sigma
0.0455 for a mean similarity of 0.5) with the code's GMM settings (reg_covar 0), the BIC minimum is at
k = 36, 40, 42 and 42 for four seeds of 5 initialisations each, and at k = 40 for the best of the 20
initialisations; k = 29 is 741 BIC units above that minimum, and the BIC over k = 30 to 44 spans only
784 units. The best of 20 is outside the 20 to 38 band (one seed alone, k = 36, would give partial).
The code's variant (100 ms PSTHs, exp(-d / 0.07975)) gives k = 33 and 35 for two seeds and 33 for the
best of 10, also outside 26 to 32. The paper's 1,000 initialisations were not run; the minimum moves with the initialisations, so this outcome is not stable.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig5c-cluster-removal | blocked (methods underspecified) |
| fig7d-context-decoding-after-whisker | blocked (methods underspecified) |
| fig4-29-clusters | not reproduced |
| fig5d-null-space-decoding | blocked (not runnable) |

**How the analysis differs from the paper's.** The code's decoder is run once per session (Fig. 5c,
as the code) in three seed sets and with the code's quiet rule, and averaged over 5 runs (Fig. 7d, as
the code); the Methods' grid search is run once, for clusters 11 and 25 only (Fig. 5c), and with two of
the ten baseline bins (Fig. 7d; a run with all ten did not finish in 53 min). For Fig. 5c a change of
outcome to not reproduced is proposed (see the record). The GMM uses 5 initialisations per k and seed (20 in all for the
Methods' variant, 10 for the code's) in place of 1,000, and the shuffle test for the number of PCs is
not run.
fig5d-null-space-decoding needs the orthogonal null and potent subspaces of the MATLAB toolbox
subspaceID (with manopt); it was not run here.

**What was found about the dataset.** The paper's cluster labels link one to one to the dataset's
units through the code's global unit index.